# PRT565 — SHAP Explainability
Explains a cost-sensitive Random Forest. Run Notebook 02 first and set `PHISHING_WEIGHT` to the cost ratio you decide to discuss.

In [ ]:
# If needed: %pip install shap pandas numpy scikit-learn matplotlib

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
DATA_DIR=Path('.')
import shap
from sklearn.ensemble import RandomForestClassifier
OUT=Path('results_xai'); OUT.mkdir(exist_ok=True)

In [ ]:
train=pd.read_csv(DATA_DIR/'train.csv'); test=pd.read_csv(DATA_DIR/'test.csv'); Xtr=train.drop(columns='label'); ytr=train['label']; Xte=test.drop(columns='label')
PHISHING_WEIGHT=5
model=RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1,class_weight={0:PHISHING_WEIGHT,1:1}); model.fit(Xtr,ytr)

In [ ]:
sample=Xte.sample(n=min(1000,len(Xte)),random_state=42); explainer=shap.TreeExplainer(model); sv=explainer(sample); phishing=sv[:,:,0] if sv.values.ndim==3 else sv
shap.plots.bar(phishing,max_display=15,show=False); plt.tight_layout(); plt.savefig(OUT/'shap_global_bar.png',dpi=200,bbox_inches='tight'); plt.show()
shap.plots.beeswarm(phishing,max_display=15,show=False); plt.tight_layout(); plt.savefig(OUT/'shap_beeswarm.png',dpi=200,bbox_inches='tight'); plt.show()

In [ ]:
imp=pd.DataFrame({'Feature':sample.columns,'Mean Absolute SHAP':np.abs(phishing.values).mean(axis=0)}).sort_values('Mean Absolute SHAP',ascending=False); display(imp.head(20)); imp.to_csv(OUT/'shap_feature_importance.csv',index=False)

**Interpretation:** SHAP explains model behaviour, not causation. Discuss which engineered features most influence phishing predictions.